# CLAHE com Seleção por CSV — Realce de Trincas e Detalhes em Fotos de Obra

Este notebook aplica **CLAHE** — *Contrast Limited Adaptive Histogram Equalization* — em imagens de obras, mas de forma **seletiva**, utilizando o relatório CSV gerado na etapa de análise exploratória.

A ideia é processar **somente as imagens indicadas no CSV** como candidatas a realce de contraste, correção de iluminação ou aplicação de CLAHE.

**Tema da aula:** Fluxo de um sistema baseado em visão computacional  
**Etapa:** Pré-processamento  
**Técnica:** CLAHE — Equalização de Histograma Adaptativa Limitada por Contraste  
**Versão:** 2.1 — Google Colab + Processamento Seletivo por CSV  
**Data:** Agosto de 2025

## 1. Objetivo do notebook

O objetivo é realçar trincas, fissuras, bordas e detalhes sutis em imagens de concreto, utilizando CLAHE.

Diferente da versão anterior, este notebook **não processa todas as imagens automaticamente**.

Ele segue o fluxo:

```text
Relatório de análise exploratória
→ leitura do CSV
→ identificação das imagens marcadas para CLAHE / realce
→ aplicação seletiva do CLAHE
→ geração de imagens, comparativos e relatório
```

Isso evita aplicar pré-processamento desnecessário em imagens que já estão adequadas.

## 2. O que é CLAHE?

CLAHE significa **Contrast Limited Adaptive Histogram Equalization**.

É uma técnica de realce de contraste local. Em vez de alterar o contraste da imagem inteira de uma vez, o CLAHE divide a imagem em pequenas regiões e melhora o contraste em cada uma delas.

Isso é útil em imagens de concreto porque trincas e fissuras podem aparecer com baixo contraste em relação à superfície.

## 3. Quando aplicar CLAHE?

O CLAHE é especialmente útil quando a análise exploratória indica problemas como:

- baixo contraste;
- iluminação irregular;
- baixa visibilidade de detalhes;
- necessidade de realce de contraste;
- necessidade de equalização de histograma;
- necessidade de destacar trincas, bordas ou fissuras.

Neste notebook, essas indicações serão buscadas no arquivo:

```text
relatorio_preprocessamento.csv
```

## 4. Estrutura esperada no Google Drive

```text
MyDrive/
└── Python_VC/
    ├── fotos_obra/
    ├── output_clahe/
    └── relatorio_preprocessamento.csv
```

A pasta `fotos_obra` contém as imagens originais.  
A pasta `output_clahe` receberá os resultados processados.  
O arquivo `relatorio_preprocessamento.csv` deve ter sido gerado pela análise exploratória.

## 5. Importação das bibliotecas

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from google.colab import drive
import pandas as pd
from pathlib import Path

## 6. Montagem do Google Drive e configuração

In [ ]:
# Montar o Google Drive
drive.mount('/content/drive')

# Configurações principais
PASTA_BASE = '/content/drive/MyDrive/Python_VC'
CSV_RELATORIO = os.path.join(PASTA_BASE, "relatorio_preprocessamento.csv")
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output_clahe")

# Parâmetros CLAHE
PARAMS_CLAHE = {
    'clip_limit': 3.0,          # Limite de contraste. Valores comuns: 2.0 a 4.0
    'tile_grid_size': (8, 8),   # Grade local para equalização
    'redimensionar_max': 1200,  # Largura máxima. Use 0 para manter original
}

# Nome da coluna que contém o nome do arquivo da imagem
COLUNA_IMAGEM = 'Imagem'

# Colunas possíveis no CSV que podem indicar necessidade de CLAHE ou realce de contraste.
# O notebook usará qualquer uma que existir no CSV.
COLUNAS_INDICATIVAS_CLAHE = [
    'CLAHE',
    'Aplicar CLAHE',
    'Equalização de Histograma',
    'Equalizacao de Histograma',
    'Realce de Contraste',
    'Melhoria de Contraste',
    'Ajuste de Contraste',
    'Correção de Iluminação',
    'Correcao de Iluminacao',
    'Baixo Contraste',
    'Contraste',
    'Realce de Detalhes',
    'Realce de Trincas'
]

# Se True, caso nenhuma coluna indicativa seja encontrada, processa todas as imagens do CSV.
# Para uso metodológico mais rigoroso, mantenha False.
PROCESSAR_TODAS_SEM_COLUNA = False

print("Configuração carregada.")
print(f"Pasta base: {PASTA_BASE}")
print(f"CSV relatório: {CSV_RELATORIO}")
print(f"Pasta de entrada: {PASTA_ENTRADA}")
print(f"Pasta de saída: {PASTA_SAIDA}")
print(f"Parâmetros CLAHE: {PARAMS_CLAHE}")

## 7. Verificação da configuração

Esta célula verifica se existem:

- pasta base;
- pasta de entrada com imagens;
- arquivo CSV da análise exploratória.

In [ ]:
def verificar_configuracao():
    """Verifica se todos os arquivos e pastas necessários existem."""
    print("🔍 VERIFICANDO CONFIGURAÇÃO:")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📊 Arquivo CSV: {CSV_RELATORIO}")
    print(f"📁 Pasta entrada: {PASTA_ENTRADA}")
    print(f"📁 Pasta saída: {PASTA_SAIDA}")
    print(f"🎛️ Parâmetros CLAHE: {PARAMS_CLAHE}")

    problemas = []

    if not os.path.exists(PASTA_BASE):
        problemas.append("❌ Pasta base não encontrada.")

    if not os.path.exists(CSV_RELATORIO):
        problemas.append("❌ Arquivo 'relatorio_preprocessamento.csv' não encontrado.")
        problemas.append("💡 Execute primeiro o notebook/script de análise exploratória.")

    if not os.path.exists(PASTA_ENTRADA):
        problemas.append("❌ Pasta de entrada 'fotos_obra' não encontrada.")

    if os.path.exists(PASTA_ENTRADA):
        arquivos_imagem = [
            f for f in os.listdir(PASTA_ENTRADA)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'))
        ]

        if not arquivos_imagem:
            problemas.append("❌ Nenhuma imagem encontrada na pasta 'fotos_obra'.")
        else:
            print(f"📸 Imagens encontradas na pasta: {len(arquivos_imagem)}")

    if problemas:
        print("\n".join(problemas))
        return False

    print("✅ Configuração verificada com sucesso.")
    return True


verificar_configuracao()

## 8. Leitura do CSV e seleção das imagens para CLAHE

O notebook lê o `relatorio_preprocessamento.csv` e procura colunas que indiquem a necessidade de CLAHE, realce de contraste ou correção de iluminação.

A coluna com o nome da imagem deve se chamar, por padrão:

```text
Imagem
```

Caso seu CSV use outro nome, ajuste a variável `COLUNA_IMAGEM`.

In [ ]:
def interpretar_booleano(valor):
    """
    Interpreta valores comuns de verdadeiro/falso vindos de CSV.

    Aceita:
    True, 'TRUE', 'True', 'Sim', 'SIM', '1', 'yes', 'x', etc.
    """
    if pd.isna(valor):
        return False

    if isinstance(valor, bool):
        return valor

    if isinstance(valor, (int, float)):
        return valor == 1

    texto = str(valor).strip().lower()

    valores_verdadeiros = {
        'true', 'verdadeiro', 'sim', 's', 'yes', 'y', '1', 'x',
        'aplicar', 'necessario', 'necessário', 'recomendado'
    }

    return texto in valores_verdadeiros


def ler_imagens_para_clahe(csv_path):
    """
    Lê o CSV de análise exploratória e identifica imagens que devem receber CLAHE.

    Retorno:
        list: lista de nomes de arquivos a processar.
        pd.DataFrame: dataframe original lido.
        list: colunas usadas como critério.
    """
    print("\n📊 LENDO RELATÓRIO CSV PARA SELEÇÃO DO CLAHE...")

    try:
        df = pd.read_csv(csv_path)

        print(f"✅ CSV carregado com {len(df)} registros.")
        print(f"📋 Colunas disponíveis: {list(df.columns)}")

        if COLUNA_IMAGEM not in df.columns:
            print(f"❌ Coluna '{COLUNA_IMAGEM}' não encontrada no CSV.")
            print("💡 Ajuste a variável COLUNA_IMAGEM para o nome correto da coluna de imagens.")
            return [], df, []

        # Identificar colunas indicativas existentes no CSV
        colunas_encontradas = [
            col for col in COLUNAS_INDICATIVAS_CLAHE
            if col in df.columns
        ]

        if not colunas_encontradas:
            print("\n⚠️ Nenhuma coluna indicativa de CLAHE/realce foi encontrada.")
            print("Colunas procuradas:")
            for col in COLUNAS_INDICATIVAS_CLAHE:
                print(f"   - {col}")

            if PROCESSAR_TODAS_SEM_COLUNA:
                print("\n⚠️ PROCESSAR_TODAS_SEM_COLUNA=True. Todas as imagens do CSV serão processadas.")
                imagens = df[COLUNA_IMAGEM].dropna().astype(str).tolist()
                return imagens, df, []

            print("\n❌ Nenhuma imagem será processada.")
            print("💡 Sugestão: inclua no CSV uma coluna como 'CLAHE', 'Realce de Contraste' ou 'Baixo Contraste'.")
            return [], df, []

        print(f"✅ Colunas usadas para seleção: {colunas_encontradas}")

        # Criar condição: qualquer coluna indicativa marcada como verdadeira
        condicao = pd.Series(False, index=df.index)

        for col in colunas_encontradas:
            condicao = condicao | df[col].apply(interpretar_booleano)

        imagens_para_processar = (
            df.loc[condicao, COLUNA_IMAGEM]
            .dropna()
            .astype(str)
            .tolist()
        )

        print("\n📈 Estatísticas da seleção:")
        print(f"   - Total de registros no CSV: {len(df)}")
        print(f"   - Imagens selecionadas para CLAHE: {len(imagens_para_processar)}")
        print(f"   - Imagens não selecionadas: {len(df) - len(imagens_para_processar)}")

        if imagens_para_processar:
            print("\n📝 Primeiras imagens selecionadas:")
            for img in imagens_para_processar[:10]:
                print(f"   ✅ {img}")

            if len(imagens_para_processar) > 10:
                print(f"   ... e mais {len(imagens_para_processar) - 10} imagens")
        else:
            print("ℹ️ Nenhuma imagem foi marcada para CLAHE no CSV.")

        return imagens_para_processar, df, colunas_encontradas

    except Exception as e:
        print(f"❌ Erro ao ler o CSV: {str(e)}")
        return [], None, []


imagens_para_clahe, df_preprocessamento, colunas_criterio_clahe = ler_imagens_para_clahe(CSV_RELATORIO)

## 9. Funções auxiliares do CLAHE

In [ ]:
def garantir_pasta(pasta):
    """Cria pasta se não existir."""
    os.makedirs(pasta, exist_ok=True)


def redimensionar_imagem(imagem, max_width):
    """Redimensiona imagem mantendo proporção."""
    if max_width and max_width > 0 and imagem.shape[1] > max_width:
        proporcao = max_width / imagem.shape[1]
        nova_largura = max_width
        nova_altura = int(imagem.shape[0] * proporcao)

        imagem_redimensionada = cv2.resize(
            imagem,
            (nova_largura, nova_altura),
            interpolation=cv2.INTER_AREA
        )

        print(
            f"   📐 Redimensionada: "
            f"{imagem.shape[1]}x{imagem.shape[0]} → {nova_largura}x{nova_altura}"
        )

        return imagem_redimensionada

    return imagem


def calcular_entropia(imagem):
    """Calcula a entropia da imagem como medida de informação visual."""
    histograma = cv2.calcHist([imagem], [0], None, [256], [0, 256])
    histograma = histograma[histograma > 0] / np.sum(histograma)

    entropia = -np.sum(histograma * np.log2(histograma))

    return entropia

## 10. Aplicação do CLAHE em uma imagem

Esta função aplica CLAHE e calcula métricas antes e depois do processamento.

In [ ]:
def aplicar_clahe_imagem(imagem, nome_arquivo):
    """Aplica CLAHE em uma imagem individual."""
    print(f"   🎯 Aplicando CLAHE: {nome_arquivo}")

    # Redimensionar se necessário
    imagem_redimensionada = redimensionar_imagem(
        imagem,
        PARAMS_CLAHE['redimensionar_max']
    )

    # Converter para escala de cinza se necessário
    if len(imagem_redimensionada.shape) == 3:
        imagem_cinza = cv2.cvtColor(imagem_redimensionada, cv2.COLOR_BGR2GRAY)
    else:
        imagem_cinza = imagem_redimensionada.copy()

    contraste_antes = np.std(imagem_cinza)
    entropia_antes = calcular_entropia(imagem_cinza)

    metricas_antes = {
        'contraste': contraste_antes,
        'brilho_medio': np.mean(imagem_cinza),
        'entropia': entropia_antes,
        'dimensoes': f"{imagem_cinza.shape[1]}x{imagem_cinza.shape[0]}"
    }

    # Aplicar CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=PARAMS_CLAHE['clip_limit'],
        tileGridSize=PARAMS_CLAHE['tile_grid_size']
    )

    imagem_clahe = clahe.apply(imagem_cinza)

    contraste_depois = np.std(imagem_clahe)
    entropia_depois = calcular_entropia(imagem_clahe)

    melhoria_contraste = (
        ((contraste_depois - contraste_antes) / contraste_antes) * 100
        if contraste_antes != 0 else 0
    )

    melhoria_entropia = (
        ((entropia_depois - entropia_antes) / entropia_antes) * 100
        if entropia_antes != 0 else 0
    )

    metricas_depois = {
        'contraste': contraste_depois,
        'brilho_medio': np.mean(imagem_clahe),
        'entropia': entropia_depois,
        'melhoria_contraste': melhoria_contraste,
        'melhoria_entropia': melhoria_entropia
    }

    print(f"   📊 Contraste: {metricas_antes['contraste']:.2f} → {metricas_depois['contraste']:.2f}")
    print(f"   📈 Melhoria de contraste: {metricas_depois['melhoria_contraste']:+.1f}%")
    print(f"   🧠 Melhoria de entropia: {metricas_depois['melhoria_entropia']:+.1f}%")

    return imagem_cinza, imagem_clahe, {
        'antes': metricas_antes,
        'depois': metricas_depois
    }

## 11. Salvamento dos resultados

Para cada imagem processada, serão salvos:

- imagem original em cinza;
- imagem após CLAHE;
- comparação simples lado a lado;
- comparativo com histogramas.

In [ ]:
def salvar_resultados_clahe(imagem_original, imagem_clahe, metricas, nome_arquivo, pasta_saida):
    """Salva os resultados do processamento CLAHE."""
    nome_base = os.path.splitext(nome_arquivo)[0]

    # Salvar imagem CLAHE
    caminho_clahe = os.path.join(pasta_saida, f"{nome_base}_clahe.png")
    cv2.imwrite(caminho_clahe, imagem_clahe)

    # Salvar imagem original em cinza
    caminho_original = os.path.join(pasta_saida, f"{nome_base}_cinza.png")
    cv2.imwrite(caminho_original, imagem_original)

    # Criar visualização comparativa
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    axes[0, 0].imshow(imagem_original, cmap='gray')
    axes[0, 0].set_title("Imagem Original em Cinza")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(imagem_clahe, cmap='gray')
    axes[0, 1].set_title("Após CLAHE")
    axes[0, 1].axis("off")

    axes[1, 0].hist(imagem_original.ravel(), 256, [0, 256], alpha=0.7, label='Original')
    axes[1, 0].set_title("Histograma — Original")
    axes[1, 0].set_xlabel("Intensidade de pixel")
    axes[1, 0].set_ylabel("Frequência")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].hist(imagem_clahe.ravel(), 256, [0, 256], alpha=0.7, label='CLAHE')
    axes[1, 1].set_title("Histograma — Após CLAHE")
    axes[1, 1].set_xlabel("Intensidade de pixel")
    axes[1, 1].set_ylabel("Frequência")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    texto_metricas = f"Clip Limit: {PARAMS_CLAHE['clip_limit']}\n"
    texto_metricas += f"Tile Grid: {PARAMS_CLAHE['tile_grid_size']}\n"
    texto_metricas += f"Dimensões: {metricas['antes']['dimensoes']}\n"
    texto_metricas += f"Melhoria Contraste: {metricas['depois']['melhoria_contraste']:+.1f}%\n"
    texto_metricas += f"Melhoria Entropia: {metricas['depois']['melhoria_entropia']:+.1f}%"

    fig.text(
        0.5,
        0.01,
        texto_metricas,
        ha='center',
        va='bottom',
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.3")
    )

    plt.tight_layout()

    caminho_comparativo = os.path.join(pasta_saida, f"{nome_base}_comparativo.png")
    plt.savefig(caminho_comparativo, dpi=150, bbox_inches='tight')
    plt.close()

    comparacao_simples = cv2.hconcat([imagem_original, imagem_clahe])
    caminho_comparacao_simples = os.path.join(
        pasta_saida,
        f"{nome_base}_comparacao_simples.jpg"
    )
    cv2.imwrite(caminho_comparacao_simples, comparacao_simples)

    caminhos = {
        'original_cinza': caminho_original,
        'clahe': caminho_clahe,
        'comparativo': caminho_comparativo,
        'comparacao_simples': caminho_comparacao_simples
    }

    return caminhos

## 12. Processamento seletivo em lote

Esta função processa somente as imagens selecionadas a partir do CSV.

In [ ]:
def processar_clahe_seletivo(lista_imagens):
    """Processa seletivamente as imagens indicadas pelo CSV."""
    print("\n🔄 INICIANDO CLAHE SELETIVO COM BASE NO CSV...")
    print("=" * 60)

    garantir_pasta(PASTA_SAIDA)

    if not lista_imagens:
        print("ℹ️ Nenhuma imagem selecionada para CLAHE.")
        return []

    resultados_gerais = []
    processadas = 0
    nao_encontradas = 0
    erros = 0

    for i, arquivo in enumerate(lista_imagens, 1):
        print(f"\n[{i}/{len(lista_imagens)}] {arquivo}")

        caminho_imagem = os.path.join(PASTA_ENTRADA, arquivo)

        if not os.path.exists(caminho_imagem):
            print(f"❌ Imagem não encontrada na pasta de entrada: {arquivo}")
            nao_encontradas += 1
            continue

        try:
            imagem = cv2.imread(caminho_imagem)

            if imagem is None:
                print(f"❌ Não foi possível carregar: {arquivo}")
                erros += 1
                continue

            imagem_original, imagem_clahe, metricas = aplicar_clahe_imagem(imagem, arquivo)

            caminhos = salvar_resultados_clahe(
                imagem_original,
                imagem_clahe,
                metricas,
                arquivo,
                PASTA_SAIDA
            )

            resultados_gerais.append({
                'imagem': arquivo,
                'dimensoes': metricas['antes']['dimensoes'],
                'contraste_antes': metricas['antes']['contraste'],
                'contraste_depois': metricas['depois']['contraste'],
                'entropia_antes': metricas['antes']['entropia'],
                'entropia_depois': metricas['depois']['entropia'],
                'melhoria_contraste': metricas['depois']['melhoria_contraste'],
                'melhoria_entropia': metricas['depois']['melhoria_entropia'],
                'colunas_criterio_csv': ", ".join(colunas_criterio_clahe),
                'caminho_original_cinza': caminhos['original_cinza'],
                'caminho_clahe': caminhos['clahe'],
                'caminho_comparativo': caminhos['comparativo'],
                'caminho_comparacao_simples': caminhos['comparacao_simples']
            })

            processadas += 1
            print(f"   ✅ CLAHE aplicado: {arquivo}")

        except Exception as e:
            print(f"❌ Erro ao processar {arquivo}: {str(e)}")
            erros += 1

    print("\n" + "=" * 60)
    print("📊 RESUMO DO CLAHE SELETIVO:")
    print(f"   ✅ Imagens processadas com sucesso: {processadas}")
    print(f"   ❌ Imagens não encontradas: {nao_encontradas}")
    print(f"   ❌ Erros de processamento: {erros}")
    print(f"   📁 Pasta de saída: {PASTA_SAIDA}")

    if resultados_gerais:
        melhoria_contraste_media = np.mean([r['melhoria_contraste'] for r in resultados_gerais])
        melhoria_entropia_media = np.mean([r['melhoria_entropia'] for r in resultados_gerais])

        print(f"   📈 Melhoria média no contraste: {melhoria_contraste_media:+.1f}%")
        print(f"   🧠 Melhoria média na entropia: {melhoria_entropia_media:+.1f}%")

    return resultados_gerais

## 13. Geração dos relatórios

O notebook gera:

- `relatorio_clahe_seletivo.csv`
- `resumo_clahe_seletivo.txt`

In [ ]:
def gerar_relatorio_clahe(resultados):
    """Gera relatório detalhado do processamento CLAHE seletivo."""
    if not resultados:
        print("ℹ️ Não há resultados para gerar relatório.")
        return None

    relatorio_path = os.path.join(PASTA_SAIDA, "relatorio_clahe_seletivo.csv")

    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(relatorio_path, index=False)

    print(f"📄 Relatório CLAHE salvo: {relatorio_path}")

    resumo_path = os.path.join(PASTA_SAIDA, "resumo_clahe_seletivo.txt")

    with open(resumo_path, 'w', encoding='utf-8') as f:
        f.write("RELATÓRIO DE PROCESSAMENTO CLAHE SELETIVO\n")
        f.write("=" * 50 + "\n")
        f.write(f"Data: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"CSV utilizado: {CSV_RELATORIO}\n")
        f.write(f"Colunas usadas como critério: {colunas_criterio_clahe}\n")
        f.write(f"Parâmetros CLAHE: {PARAMS_CLAHE}\n")
        f.write(f"Total de imagens processadas: {len(resultados)}\n")

        melhoria_contraste_media = np.mean([r['melhoria_contraste'] for r in resultados])
        melhoria_entropia_media = np.mean([r['melhoria_entropia'] for r in resultados])

        f.write("\nESTATÍSTICAS GERAIS:\n")
        f.write(f"  Melhoria média no contraste: {melhoria_contraste_media:+.1f}%\n")
        f.write(f"  Melhoria média na entropia: {melhoria_entropia_media:+.1f}%\n")

        f.write("\nDETALHES POR IMAGEM:\n")

        for res in resultados:
            f.write(
                f"- {res['imagem']}: "
                f"contraste {res['melhoria_contraste']:+.1f}%, "
                f"entropia {res['melhoria_entropia']:+.1f}%\n"
            )

    print(f"📋 Resumo CLAHE salvo: {resumo_path}")

    return df_resultados

## 14. Visualização de exemplos

In [ ]:
def mostrar_exemplos_resultados(num_exemplos=3):
    """Mostra exemplos dos resultados gerados."""
    if not os.path.exists(PASTA_SAIDA):
        print("ℹ️ Pasta de saída ainda não existe.")
        return

    arquivos_comparativos = [
        f for f in os.listdir(PASTA_SAIDA)
        if f.endswith('_comparativo.png')
    ]

    if not arquivos_comparativos:
        print("ℹ️ Nenhum arquivo comparativo encontrado para exibir.")
        return

    exemplos = arquivos_comparativos[:num_exemplos]

    print(f"\n🖼️ EXIBINDO {len(exemplos)} EXEMPLOS DE CLAHE:")

    for exemplo in exemplos:
        caminho_exemplo = os.path.join(PASTA_SAIDA, exemplo)

        try:
            img = plt.imread(caminho_exemplo)

            plt.figure(figsize=(15, 10))
            plt.imshow(img)
            plt.title(f"Exemplo CLAHE: {exemplo}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"   ❌ Erro ao exibir {exemplo}: {str(e)}")

## 15. Execução completa

Execute esta célula para rodar o fluxo completo:

```text
Verificação
→ leitura do CSV
→ seleção das imagens
→ aplicação do CLAHE
→ relatório
→ exemplos visuais
```

In [ ]:
def main():
    """Função principal do notebook."""
    print("🚀 INICIANDO CLAHE SELETIVO — REALCE DE TRINCAS E DETALHES")
    print("=" * 60)

    if not verificar_configuracao():
        print("\n❌ Não foi possível iniciar o processamento.")
        print("\n💡 SOLUÇÃO:")
        print("1. Verifique se a pasta 'Python_VC/fotos_obra' existe no seu Drive.")
        print("2. Verifique se há imagens na pasta 'fotos_obra'.")
        print("3. Execute primeiro a análise exploratória para gerar o CSV.")
        print("4. Confirme se o arquivo 'relatorio_preprocessamento.csv' existe na pasta Python_VC.")
        return []

    imagens_para_processar, df_csv, colunas_usadas = ler_imagens_para_clahe(CSV_RELATORIO)

    if not imagens_para_processar:
        print("\nℹ️ Nenhuma imagem será processada com CLAHE.")
        print("💡 Verifique se o CSV possui uma coluna indicativa, como 'CLAHE', 'Realce de Contraste' ou 'Baixo Contraste'.")
        return []

    resultados = processar_clahe_seletivo(imagens_para_processar)

    if resultados:
        df_resultados = gerar_relatorio_clahe(resultados)

        print("\n🎯 CLAHE SELETIVO CONCLUÍDO!")
        print(f"   📁 Resultados salvos em: {PASTA_SAIDA}")
        print(f"   📊 {len(resultados)} imagens processadas com sucesso")
        print("   🔍 Contraste realçado para melhor visualização de trincas e detalhes")

        mostrar_exemplos_resultados(2)

        return resultados

    print("\n❌ Nenhuma imagem foi processada com sucesso.")
    return []


resultados_clahe = main()

## 16. Visualizar tabela do relatório

In [ ]:
if 'resultados_clahe' in globals() and resultados_clahe:
    df_clahe = pd.DataFrame(resultados_clahe)
    display(df_clahe)
else:
    print("Execute primeiro a célula de processamento completo.")

## 17. Ajustes de critério do CSV

Se o seu CSV tiver uma coluna com outro nome para indicar CLAHE, adicione o nome da coluna na lista abaixo antes de executar novamente:

```python
COLUNAS_INDICATIVAS_CLAHE.append('Nome da sua coluna')
```

Exemplo:

```python
COLUNAS_INDICATIVAS_CLAHE.append('Aplicar CLAHE')
```

In [ ]:
# Exemplo de ajuste, caso seu CSV use outro nome de coluna:
# COLUNAS_INDICATIVAS_CLAHE.append('Aplicar CLAHE')
# COLUNAS_INDICATIVAS_CLAHE.append('Necessita Realce')

COLUNAS_INDICATIVAS_CLAHE

## 18. Ajustes dos parâmetros CLAHE

Se o efeito estiver muito forte, reduza o `clip_limit`:

```python
PARAMS_CLAHE['clip_limit'] = 2.0
```

Se o efeito estiver fraco, aumente o `clip_limit`:

```python
PARAMS_CLAHE['clip_limit'] = 4.0
```

Para detalhes mais locais, teste outra grade:

```python
PARAMS_CLAHE['tile_grid_size'] = (16, 16)
```

Para preservar resolução original:

```python
PARAMS_CLAHE['redimensionar_max'] = 0
```

In [ ]:
# Ajustes opcionais dos parâmetros

# PARAMS_CLAHE['clip_limit'] = 2.0
# PARAMS_CLAHE['tile_grid_size'] = (8, 8)
# PARAMS_CLAHE['redimensionar_max'] = 1200

PARAMS_CLAHE

## 19. Interpretação técnica dos resultados

Ao avaliar os comparativos, observe:

1. **As trincas ficaram mais visíveis?**  
   Esse é o principal objetivo do CLAHE neste contexto.

2. **O ruído também foi realçado?**  
   Se sim, talvez seja necessário aplicar redução de ruído antes.

3. **A imagem ficou artificial demais?**  
   Reduza o `clip_limit`.

4. **A melhoria de contraste foi positiva?**  
   O relatório apresenta a variação percentual de contraste.

5. **A entropia aumentou?**  
   Em geral, aumento de entropia indica mais informação visual, mas também pode indicar mais ruído.

O CLAHE deve ser validado visualmente antes de entrar como etapa fixa em um pipeline de visão computacional.

## 20. Fluxo recomendado

Um fluxo típico para imagens de concreto pode ser:

```text
Imagem da obra
→ análise exploratória
→ seleção por CSV
→ redimensionamento, se necessário
→ redução de ruído, se necessário
→ CLAHE seletivo
→ detecção de bordas
→ YOLO ou U-Net
→ validação técnica
→ relatório
```

Esta versão do notebook se encaixa exatamente na etapa:

```text
seleção por CSV → CLAHE seletivo
```